In [1]:
from utils import get_svamp_df, get_gsm8k
from statistical_significance import compare_accuracies, compare_accuracies_v2, compare_accuracies_v3

import pandas as pd
import numpy as np

df = get_gsm8k()
df.head()

GSM8K
gemma-3-27b-it.xlsx
qwen3_1.7b.xlsx
smollm2_360m.xlsx
qwen2-math_1.5b.xlsx
qwen3_0.6b.xlsx
qwen2.5_0.5b.xlsx
gemma3_1b.xlsx
deepseek-r1_1.5b.xlsx
llama3.2_1b.xlsx
DP-Reflection
qwen3_1.7b.xlsx
qwen3_0.6b.xlsx
qwen2.5_0.5b.xlsx
deepseek-r1_1.5b.xlsx
ReAct
qwen3_1.7b.xlsx
qwen3_0.6b.xlsx
qwen2.5_0.5b.xlsx
llama3.2_1b.xlsx
ReWOO
gemma-3-27b-it.xlsx
qwen3_1.7b.xlsx
smollm2_360m.xlsx
qwen2-math_1.5b.xlsx
qwen3_0.6b.xlsx
qwen2.5_0.5b.xlsx
gemma3_1b.xlsx
deepseek-r1_1.5b.xlsx
llama3.2_1b.xlsx


,Dataset,Method,Model,question,target_answer,response,is_correct,input_tokens,output_tokens,total_tokens,reasoning,Error Class,Error Type
0,GSM8K,Direktes Prompting,gemma-3-27b-it,Janet‚Äôs ducks lay 16 eggs per day. She eats ...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...,Let $E$ be the number of eggs Janet's ducks la...,True,NaN,NaN,NaN,Let $E$ be the number of eggs Janet's ducks la...,NaN,NaN
1,GSM8K,Direktes Prompting,gemma-3-27b-it,A robe takes 2 bolts of blue fiber and half th...,It takes 2/2=<<2/2=1>>1 bolt of white fiber\nS...,Let the number of bolts of blue fiber be $B$ a...,True,NaN,NaN,NaN,Let the number of bolts of blue fiber be $B$ a...,NaN,NaN
2,GSM8K,Direktes Prompting,gemma-3-27b-it,Josh decides to try flipping a house. He buys...,The cost of the house and repairs came out to ...,Let the initial cost of the house be $C = \$80...,True,NaN,NaN,NaN,Let the initial cost of the house be $C = \$80...,NaN,NaN
3,GSM8K,Direktes Prompting,gemma-3-27b-it,James decides to run 3 sprints 3 times a week....,He sprints 3*3=<<3*3=9>>9 times\nSo he runs 9*...,Let $s$ be the number of sprints James runs ea...,True,NaN,NaN,NaN,Let $s$ be the number of sprints James runs ea...,NaN,NaN
4,GSM8K,Direktes Prompting,gemma-3-27b-it,"Every day, Wendi feeds each of her chickens th...","If each chicken eats 3 cups of feed per day, t...",Let $n$ be the number of chickens Wendi has. W...,True,NaN,NaN,NaN,Let $n$ be the number of chickens Wendi has. W...,NaN,NaN


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import ollama
from tqdm import tqdm

def embed(txt: str):
    emb = ollama.embed(
        model="all-minilm:33m",
        input=txt
    )
    return emb.embeddings[0]

cache = {}

for _, row in tqdm(df.iterrows()):
    if row["target_answer"] not in cache:
        embeddings = embed(row["target_answer"])
        cache[row["target_answer"]] = embeddings

print(len(cache))

7800it [00:08, 958.50it/s] 

324


In [23]:
import numpy as np
def similarity(row):
    try:
        target_answer_emb = cache[row["target_answer"]]
        reasoning_emb = embed(row["reasoning"])
        return cosine_similarity(
            [target_answer_emb], 
            [reasoning_emb]
        )[0][0]
    except:
        return np.nan

df["similarity"] = df.apply(similarity, axis=1)

In [24]:
df_tmp = df.groupby(["Model", "Method"]).agg({
    "similarity": "mean",
    "is_correct": "mean"
}).reset_index().sort_values(by="similarity", ascending=False)

df_tmp

,Model,Method,similarity,is_correct
7,llama3.2:1b,Direktes Prompting,0.826081,0.506667
10,qwen2-math:1.5b,Direktes Prompting,0.821315,0.823333
11,qwen2-math:1.5b,ReWOO,0.811584,0.003333
13,qwen2.5:0.5b,Direktes Prompting,0.811337,0.403333
3,gemma-3-27b-it,Direktes Prompting,0.809454,0.973333
17,qwen3:0.6b,Direktes Prompting,0.808806,0.610000
1,deepseek-r1:1.5b,Direktes Prompting,0.804841,0.733333
5,gemma3:1b,Direktes Prompting,0.803130,0.586667
24,smollm2:360m,Direktes Prompting,0.802818,0.053333
12,qwen2.5:0.5b,DP-Reflection,0.798927,0.350000


In [25]:
# correlation between similarity and correctness
corr = df_tmp["similarity"].corr(df_tmp["is_correct"])
print("Correlation:", corr)

Correlation: 0.43275639191082355


In [26]:
# correlation between similarity and correctness
for m in df_tmp["Method"].unique():
    df_tmp2 = df_tmp[df_tmp["Method"] == m]
    corr = df_tmp2["similarity"].corr(df_tmp2["is_correct"])
    print(f"Correlation for {m}:", corr)

Correlation for Direktes Prompting: 0.11388689060021519
Correlation for ReWOO: 0.2449617869243061
Correlation for DP-Reflection: -0.953815784485669
Correlation for ReAct: 0.9573248771320559


In [27]:
def corr_per_method(group):
    return group["similarity"].corr(group["is_correct"])

corrs = df.groupby("Method").apply(corr_per_method)
print(corrs)

Method
DP-Reflection        -0.058801
Direktes Prompting    0.097548
ReAct                 0.344361
ReWOO                 0.068220
dtype: float64


/var/folders/rl/3ygx2m4n1w747bxzh4qvfjs00000gn/T/ipykernel_43406/771489780.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  corrs = df.groupby("Method").apply(corr_per_method)


In [28]:
df_tmp.sort_values(by=["Method", "is_correct"], ascending=False)

,Model,Method,similarity,is_correct
4,gemma-3-27b-it,ReWOO,0.737021,0.890000
2,deepseek-r1:1.5b,ReWOO,0.764112,0.773333
23,qwen3:1.7b,ReWOO,0.762617,0.743333
19,qwen3:0.6b,ReWOO,0.761092,0.553333
15,qwen2.5:0.5b,ReWOO,0.740361,0.123333
6,gemma3:1b,ReWOO,0.698735,0.113333
9,llama3.2:1b,ReWOO,0.655116,0.040000
25,smollm2:360m,ReWOO,0.742472,0.023333
11,qwen2-math:1.5b,ReWOO,0.811584,0.003333
22,qwen3:1.7b,ReAct,0.782208,0.853333
